# 01. Baseline 데이터 점검 및 기준 모델

이 노트북은 해설서를 사용하지 않는 첫 번째 기준 실험입니다.
현재 단계에서는 사례집만 사용하며, DummyClassifier를 통해 최소 기준 성능을 측정합니다.

In [ ]:
!pip -q install pandas numpy scikit-learn openpyxl

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

from google.colab import files
uploaded = files.upload()
DATA_PATH = next(iter(uploaded))
print("Loaded:", DATA_PATH)

In [ ]:
INPUT_COLUMNS = [
    "Product Name",
    "Main Ingredients (KR)",
    "Main Ingredients (EN)",
    "Product Form and Packaging",
    "Intended Use",
]
TARGET_COLUMN = "HS Code"

df = pd.read_excel(DATA_PATH, sheet_name="사례_전처리 완료")
required = INPUT_COLUMNS + [TARGET_COLUMN]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f"필수 컬럼 누락: {missing_columns}")

for col in INPUT_COLUMNS:
    df[col] = df[col].fillna("").astype(str).str.strip()

df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(str).str.strip()
df = df[df[TARGET_COLUMN].ne("")].reset_index(drop=True)

df["model_text"] = (
    "[PRODUCT] " + df["Product Name"] +
    " [INGREDIENTS_KR] " + df["Main Ingredients (KR)"] +
    " [INGREDIENTS_EN] " + df["Main Ingredients (EN)"] +
    " [FORM] " + df["Product Form and Packaging"] +
    " [USE] " + df["Intended Use"]
)

df["duplicate_group"] = pd.util.hash_pandas_object(
    df[INPUT_COLUMNS], index=False
).astype(str)

print("데이터 크기:", df.shape)
display(df[INPUT_COLUMNS + [TARGET_COLUMN]].head())

## 원본 데이터 중복·그룹 구조 점검

동일한 입력 정보가 여러 HS 코드에 연결되는 충돌 사례가 있는지 먼저 확인합니다. 동일 입력이 여러 행에 반복된 경우에는 같은 그룹으로 묶습니다. 이후 Fold 배정이 가능한지 직접 검사합니다.

In [ ]:
conflict_table = (
    df.groupby("duplicate_group")[TARGET_COLUMN]
    .nunique()
    .rename("n_hs_codes")
    .reset_index()
)
conflicting_groups = conflict_table[conflict_table["n_hs_codes"] > 1]
print("동일 입력에 복수 HS 코드가 연결된 그룹:", len(conflicting_groups))
display(conflicting_groups.head(20))

## 평가 대상 클래스 선정 및 Fold 가능성 검증

주 평가에는 클래스별 서로 다른 입력 그룹이 최소 4개 있는 경우만 포함합니다. 클래스가 각 Fold의 학습·평가 집합에 가능한 한 모두 나타나는지 사전 검사합니다. Fold 생성이 불가능하면 임의로 진행하지 않고 오류를 표시합니다.

In [ ]:
class_counts = df[TARGET_COLUMN].value_counts()
eligible_labels = class_counts[class_counts >= 4].index
eval_df = df[df[TARGET_COLUMN].isin(eligible_labels)].copy()

group_counts = eval_df.groupby(TARGET_COLUMN)["duplicate_group"].nunique()
eligible_labels = group_counts[group_counts >= 4].index
eval_df = eval_df[eval_df[TARGET_COLUMN].isin(eligible_labels)].reset_index(drop=True)

print("전체 데이터:", len(df), "건 /", df[TARGET_COLUMN].nunique(), "개 코드")
print("주 평가 데이터:", len(eval_df), "건 /", eval_df[TARGET_COLUMN].nunique(), "개 코드")
print("별도 분석 데이터:", len(df) - len(eval_df), "건")
display(eval_df[TARGET_COLUMN].value_counts().rename("count").to_frame())

if eval_df.empty:
    raise ValueError("평가 가능한 클래스가 없습니다.")

X = eval_df["model_text"]
y = eval_df[TARGET_COLUMN]
groups = eval_df["duplicate_group"]
cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
splits = list(cv.split(X, y, groups))

fold_checks = []
for fold, (train_idx, test_idx) in enumerate(splits, start=1):
    train_labels = set(y.iloc[train_idx])
    test_labels = set(y.iloc[test_idx])
    fold_checks.append({
        "fold": fold,
        "train_n": len(train_idx),
        "test_n": len(test_idx),
        "train_classes": len(train_labels),
        "test_classes": len(test_labels),
        "missing_train_classes": sorted(set(y) - train_labels),
        "missing_test_classes": sorted(set(y) - test_labels),
        "group_overlap": len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))
    })

fold_check_df = pd.DataFrame(fold_checks)
display(fold_check_df)
if any(row["group_overlap"] != 0 for row in fold_checks):
    raise ValueError("동일 입력 그룹이 학습·평가 Fold에 중복 배치되었습니다.")

## Dummy baseline: 최빈 클래스 예측

DummyClassifier는 제품 정보를 학습하지 않고 학습 Fold에서 가장 많이 등장한 HS 코드만 예측합니다.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

fold_results = []
for fold, (train_idx, test_idx) in enumerate(splits, start=1):
    model = DummyClassifier(strategy="most_frequent")
    model.fit(X.iloc[train_idx].to_frame(), y.iloc[train_idx])
    pred = model.predict(X.iloc[test_idx].to_frame())
    fold_results.append({
        "fold": fold,
        "train_n": len(train_idx),
        "test_n": len(test_idx),
        "accuracy": accuracy_score(y.iloc[test_idx], pred),
        "macro_f1": f1_score(y.iloc[test_idx], pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y.iloc[test_idx], pred, average="weighted", zero_division=0),
    })

baseline_results = pd.DataFrame(fold_results)
display(baseline_results)
display(baseline_results[["accuracy", "macro_f1", "weighted_f1"]].agg(["mean", "std"]))

In [ ]:
baseline_results.to_csv("baseline_dummy_cv_results.csv", index=False, encoding="utf-8-sig")
label_table.to_csv("hs_code_data_audit.csv", index=False, encoding="utf-8-sig")
fold_check_df.to_csv("baseline_fold_check.csv", index=False, encoding="utf-8-sig")
print("결과 CSV 저장 완료")